# 7.01 Web Search 도구 비교 — Tavily · DuckDuckGo · SerpAPI · Serper · Brave · Exa · You.com

에이전트가 **실시간 웹 정보**에 닿으려면 검색 API 도구가 필요하다. LangChain 은 7개 이상의 벤더를 동일한 `BaseTool` 인터페이스로 감싸 제공한다. 이 노트북은 **하나의 질의를 여러 벤더에 태워 응답 포맷·커버리지·비용을 비교**하는 데 집중한다.

**기본 벤더**: Tavily (LLM-친화적 요약, `.env` 의 `TAVILY_API_KEY` 이미 준비). 나머지는 key-gated 섹션.

## 학습 목표

- `TavilySearch` (v1 공식 독립 패키지 `langchain-tavily`) 사용법
- 키 없이 동작하는 `DuckDuckGoSearchRun` / `DuckDuckGoSearchResults` 경로
- `SerpAPIWrapper` · `GoogleSerperAPIWrapper` · `BraveSearch` · `ExaSearchResults` · `YouSearchTool` 의 **초기화 패턴** 과 환경변수
- 각 벤더의 **응답 구조** (dict vs 텍스트 vs Document), **요금·한도**, **특화 기능** (뉴스 모드·도메인 필터·시맨틱 검색 등)
- 에이전트에 여러 검색 도구를 물려 **토픽별 라우팅**

## 언제 쓰나

- 최신 뉴스·업데이트 (모델 knowledge cutoff 이후) — Tavily 또는 Serper 추천
- **지출 제로** + 프로토타입 — DuckDuckGo
- 시맨틱 신경망 검색 — Exa
- Bing 대안 프라이버시 검색 — Brave

## 7.01.1 환경 설정

`TAVILY_API_KEY` 가 이 노트북의 주축. DDG 섹션은 키 없이 동작. 나머지 벤더는 해당 환경변수가 있을 때만 실행.

In [ ]:
# !pip install -U langchain langchain-community langchain-tavily ddgs

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY 필요 (이 노트북의 주축)"

# 벤더별 키 존재 여부 — 아래 섹션에서 조건부 실행
vendor_keys = {
    "Tavily":  bool(os.environ.get("TAVILY_API_KEY")),
    "SerpAPI": bool(os.environ.get("SERPAPI_API_KEY")),
    "Serper":  bool(os.environ.get("SERPER_API_KEY")),
    "Brave":   bool(os.environ.get("BRAVE_SEARCH_API_KEY")),
    "Exa":     bool(os.environ.get("EXA_API_KEY")),
    "You":     bool(os.environ.get("YDC_API_KEY")) or bool(os.environ.get("YOU_API_KEY")),
}
print(vendor_keys)

## 7.01.2 기본 사용 — `TavilySearch.invoke(...)`

`TavilySearch` 는 **dict** 를 반환 (`{"query", "results", "answer", "images", ...}`). 각 `results` 원소에 `title`·`url`·`content`·`score` 가 들어 있다. `include_answer=True` 를 켜면 LLM 요약 응답을 **한 줄** 추가로 받는다.

In [ ]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(
    max_results=3,
    search_depth="basic",          # basic | advanced (비용·품질 tradeoff)
    topic="general",               # general | news | finance
    include_answer=True,
)
res = tavily.invoke({"query": "LangChain v1 release highlights"})
print("keys :", list(res.keys()))
print("answer:", res.get("answer"))
for r in res["results"]:
    print("-", r["title"], "|", r["url"], "| score=", r.get("score"))

## 7.01.3 DuckDuckGo — 키 없이 동작하는 프로토타입 옵션

`DuckDuckGoSearchRun` 은 텍스트 스니펫 문자열을, `DuckDuckGoSearchResults` 는 메타가 붙은 리스트를 반환. v1 시점에는 내부 백엔드가 `ddgs` 패키지(구 `duckduckgo-search` 에서 리네임)를 사용한다.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

ddg_run = DuckDuckGoSearchRun()
print("--- DuckDuckGoSearchRun (str) ---")
print(ddg_run.invoke("LangChain v1 release notes")[:400])

ddg_res = DuckDuckGoSearchResults(num_results=3)
print("\n--- DuckDuckGoSearchResults (list) ---")
print(ddg_res.invoke("LangChain v1 release notes")[:400])

## 7.01.4 다른 벤더들 — key-gated 초기화

각 벤더의 **가장 작은 실행 가능 예제**. 키가 없으면 import · 설명만 표시하고 스킵.

### SerpAPI · Google Serper
둘 다 Google 검색 결과를 JSON 으로 반환하는 프록시 API. SerpAPI 는 **풍부한 파싱**, Serper 는 **초저가 + 빠른 응답**.

In [ ]:
from langchain_community.utilities import SerpAPIWrapper, GoogleSerperAPIWrapper
from langchain_community.tools.google_serper.tool import GoogleSerperRun
from langchain_core.tools import Tool

if vendor_keys["SerpAPI"]:
    serp = SerpAPIWrapper()
    serp_tool = Tool(name="serpapi_search", func=serp.run,
                     description="SerpAPI 경유 구글 검색 상위 결과 요약")
    print("[SerpAPI]", serp_tool.invoke("LangChain v1 release")[:300])
else:
    print("SerpAPI 키 없음 → 스킵. (SERPAPI_API_KEY 필요, 250 req/month 무료)")

if vendor_keys["Serper"]:
    serper_tool = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())
    print("[Serper]", serper_tool.invoke("LangChain v1 release")[:300])
else:
    print("Serper 키 없음 → 스킵. (SERPER_API_KEY 필요, 2,500 req/month 무료)")

### Brave Search · Exa · You.com

- **Brave**: 프라이버시 친화, `BraveSearch.from_api_key(...)` 팩토리.
- **Exa**: 시맨틱 임베딩 기반 — "~와 비슷한 글" 검색. `langchain-exa` 패키지 별도.
- **You.com**: RAG 친화 스니펫 + 뉴스 모드. `YOU_API_KEY` 또는 `YDC_API_KEY`.

In [ ]:
if vendor_keys["Brave"]:
    from langchain_community.tools import BraveSearch
    brave = BraveSearch.from_api_key(
        api_key=os.environ["BRAVE_SEARCH_API_KEY"], search_kwargs={"count": 3}
    )
    print("[Brave]", brave.invoke("LangChain v1 release")[:300])
else:
    print("Brave 키 없음 → 스킵. (BRAVE_SEARCH_API_KEY)")

if vendor_keys["Exa"]:
    try:
        from langchain_exa import ExaSearchResults
        exa = ExaSearchResults(api_key=os.environ["EXA_API_KEY"])
        print("[Exa]", exa.invoke({"query": "LangChain v1 release", "num_results": 3}))
    except ModuleNotFoundError:
        print("`langchain-exa` 미설치 → `uv pip install langchain-exa`")
else:
    print("Exa 키 없음 → 스킵. (EXA_API_KEY)")

if vendor_keys["You"]:
    from langchain_community.tools.you import YouSearchTool
    from langchain_community.utilities.you import YouSearchAPIWrapper
    you = YouSearchTool(api_wrapper=YouSearchAPIWrapper(num_web_results=3))
    print("[You]", you.invoke("LangChain v1 release"))
else:
    print("You.com 키 없음 → 스킵. (YDC_API_KEY 또는 YOU_API_KEY)")

## 7.01.5 `create_agent` 에 연결 — 검색 도구 라우팅

2개 도구를 한 에이전트에 물려, **모델이 질문 성격에 따라 도구를 선택**하게 한다. Tavily 는 뉴스/최신성, DDG 는 무료 폴백. `description` 이 라우팅의 핵심.

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain.agents import create_agent

    tavily_news = TavilySearch(max_results=3, topic="news", search_depth="basic")
    tavily_news.name = "tavily_news"
    tavily_news.description = "최근 며칠 내 뉴스·업데이트를 찾을 때 우선 사용."

    ddg_run.name = "ddg_fallback"
    ddg_run.description = "Tavily 쿼터 초과 시 폴백용 무료 웹 검색."

    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=[tavily_news, ddg_run],
        system_prompt="웹 검색으로 근거를 수집한 뒤, 출처 URL 1~2개를 인용해 2~3문장으로 답한다.",
    )
    res = agent.invoke({"messages": [{"role": "user", "content": "LangChain v1.2 에서 바뀐 주요 import 경로를 한 줄로 알려줘"}]})
    print(res["messages"][-1].content[:600])
else:
    print("OPENAI_API_KEY 없음 → 에이전트 예제 스킵. 도구 동작은 7.01.2~4에서 확인됨.")

## 7.01.6 주요 옵션 · 비교 표

| 벤더 | 패키지 | 무료 한도 (참고) | 반환 형태 | 특화 기능 |
|------|--------|------------------|----------|----------|
| **Tavily** | `langchain-tavily` | 1,000 req/mo | `dict` (`results`, `answer`, `images`) | `topic="news"` · `include_answer` · 도메인 필터 |
| **DuckDuckGo** | `langchain-community` + `ddgs` | **무제한** (공손 요청) | `str` 또는 `list[dict]` | 키 불필요, 프로토타입 |
| **SerpAPI** | `langchain-community` | 250 req/mo | `str` | Google·Bing·Yahoo 멀티 엔진 |
| **Google Serper** | `langchain-community` | 2,500 req/mo | `str` | 가장 저렴, 빠른 응답 |
| **Brave** | `langchain-community` | 2,000 req/mo | `str` | 프라이버시 우선, 자체 인덱스 |
| **Exa** | `langchain-exa` | 1,000 req/mo | `ExaSearchResults` | 시맨틱·neural 검색 |
| **You.com** | `langchain-community` | 호출 수 제한 | `str` | RAG 친화 스니펫, 뉴스 모드 |

> 한도·가격은 각 벤더 공식 페이지에서 최신값을 확인. 위 숫자는 2026-04 기준 참고치.

### 응답 shape 차이

- Tavily `dict` — `.invoke()` 결과에서 `results[i]['content']`, `answer` 를 바로 쓸 수 있음
- DDG / SerpAPI / Serper — 대부분 **텍스트 스니펫 결합 문자열**. 파싱이 필요하면 `...Results` 클래스 변종 사용
- Exa — `List[Document]` 반환 (`ExaSearchResults`)

에이전트에 물릴 때는 **반환 포맷이 model-friendly 한가**가 관건. Tavily 가 LLM용으로 조율된 포맷이라 통합 비용이 가장 낮다.

## 7.01.7 (선택) HITL 결합 — 외부 검색 승인

공개 웹 검색 자체는 보통 승인 대상이 아니지만, 다음 두 시나리오에서는 HITL을 끼워 둘 가치가 있다.
- 사내 문서만 허용하는 정책에서 **외부 검색** 이 발생할 때 알림성 승인
- **`include_raw_content=True`** 로 긴 HTML을 받아 컨텍스트를 크게 늘리는 호출을 사람이 승인

```python
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[tavily_news],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={"tavily_news": {"allowed_decisions": ["approve", "edit", "reject"]}},
    )],
)
```

HITL resume 흐름은 `02_langchain/07_hitl_and_runtime` 참고.

## 체크리스트

- [ ] Tavily `results[i]['score']` 는 **검색엔진 관련성** — 필터·정렬에 활용 가능
- [ ] DDG 는 CI 에서 간헐 실패 — 실제 서비스에서는 **Tavily 주 + DDG 폴백** 구성 추천
- [ ] SerpAPI/Serper 는 **지역·언어 파라미터**(`gl`, `hl`) 지원 — 한국 시장 검색엔 `gl="kr", hl="ko"`
- [ ] Exa 는 **시맨틱 유사 문서** 검색 — 키워드보다 "~와 비슷한 사례" 질문에 강점
- [ ] 검색 도구 결과의 **HTML 스크랩** 이 필요하면 `Playwright` 토이킷(7.04) 을 이어 붙이는 파이프라인

## 다음

- `07_integration/07_tools/02_code_execution.ipynb` — 검색 결과를 받아 계산/변환
- `07_integration/05_retrievers/04_web_wiki_arxiv.ipynb` — retriever 인터페이스(Document 반환) 변형

## 참고

- Tavily: https://tavily.com/
- Google Serper: https://serper.dev/
- Brave Search: https://brave.com/search/api/
- Exa: https://exa.ai/
- You.com: https://you.com/api